# Ejercicio 3.2 · Hybrid search sobre el pipeline de los PDF

El índice es el de 3.1: 2.698 fragmentos de la documentación de Python 3.13, vectores de `qwen3-embedding:0.6b` en Chroma. No se vuelve a embeber el corpus y no hay API de pago.

Se comparan tres recuperadores sobre las mismas preguntas: solo denso, solo BM25 y la mezcla. La mezcla de LangChain es Reciprocal Rank Fusion con pesos. Qdrant, en local, hace la misma fusión con un vector denso y un vector disperso.


## 1. Los mismos fragmentos

Chroma ya tiene el texto, los metadatos y el vector. Se les añade `chunk_id` para que denso y BM25 hablen de los mismos identificadores. BM25 parte en minúsculas y se queda con letras, números y guiones bajos, así que `__hash__` y `id_nubmer` siguen siendo un token.


In [1]:
import os
import re
import time
from pathlib import Path

os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import chromadb
from langchain_chroma import Chroma
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_core.retrievers import BaseRetriever
from pydantic import ConfigDict, Field
from rank_bm25 import BM25Okapi


class BM25Retriever(BaseRetriever):
    """BM25 local. La misma forma que el de langchain-community, sin ese paquete."""

    vectorizer: object = None
    docs: list[Document] = Field(default_factory=list, repr=False)
    k: int = 4
    preprocess_func: object = None
    model_config = ConfigDict(arbitrary_types_allowed=True)

    @classmethod
    def from_documents(cls, documents, *, preprocess_func, **kwargs):
        lista = list(documents)
        vectorizer = BM25Okapi([preprocess_func(doc.page_content) for doc in lista])
        return cls(vectorizer=vectorizer, docs=lista, preprocess_func=preprocess_func, **kwargs)

    def _get_relevant_documents(self, query: str, *, run_manager) -> list[Document]:
        return list(self.vectorizer.get_top_n(self.preprocess_func(query), self.docs, n=self.k))

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings
from qdrant_client import QdrantClient, models

RAIZ = Path(".").resolve()
CHROMA_DIR = RAIZ / "chroma_db"
QDRANT_DIR = RAIZ / "qdrant_db"
COLECCION = "python_pdf_500"
COLECCION_QDRANT = "python_pdf_hybrid"
MODELO_EMBEDDING = "qwen3-embedding:0.6b"
MODELO_LLM = "llama3.2"
PREFIJO_QUERY = (
    "Instruct: Given a question about Python, retrieve the relevant documentation passage\n"
    "Query:"
)
K = 5
PROFUNDIDAD = 20
C_RRF = 60


def tokenizar(texto: str) -> list[str]:
    return re.findall(r"[a-z0-9_]+", texto.lower())


cliente = chromadb.PersistentClient(path=str(CHROMA_DIR))
cruda = cliente.get_collection(COLECCION)
guardado = cruda.get(include=["documents", "metadatas"])
if "chunk_id" not in guardado["metadatas"][0]:
    cruda.update(
        ids=guardado["ids"],
        metadatas=[{**meta, "chunk_id": identificador} for identificador, meta in zip(guardado["ids"], guardado["metadatas"])],
    )
    guardado = cruda.get(include=["documents", "metadatas"])

documentos = [
    Document(page_content=texto, metadata=meta, id=identificador)
    for identificador, texto, meta in zip(guardado["ids"], guardado["documents"], guardado["metadatas"])
]
por_id = {doc.metadata["chunk_id"]: doc for doc in documentos}
print(f"{len(documentos)} fragmentos. Ejemplo de id: {documentos[0].metadata['chunk_id'][:8]}")


2698 fragmentos. Ejemplo de id: 436564f2


## 2. Denso, BM25 y la mezcla

El denso es el mismo recuperador de 3.1, con el prefijo de instrucción de `qwen3-embedding` en la pregunta. BM25 es léxico: gana cuando la pregunta trae el token raro. `EnsembleRetriever` suma `peso / (rango + 60)`. El primer peso es el denso y el segundo es BM25. Cada uno aporta 20 candidatos y la tabla se queda con los 5 primeros de la fusión. Si los dos devolvieran solo 5, un fragmento que está sexto en los dos no podría entrar.


In [2]:
class EmbeddingsLocales(Embeddings):
    def __init__(self, modelo: str) -> None:
        self._interno = OllamaEmbeddings(model=modelo, keep_alive=1800, client_kwargs={"timeout": 120})
        self.cache: dict[str, list[float]] = {}

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self._interno.embed_documents(texts)

    def embed_query(self, text: str) -> list[float]:
        if text not in self.cache:
            self.cache[text] = self._interno.embed_query(f"{PREFIJO_QUERY} {text}")
        return self.cache[text]


embeddings = EmbeddingsLocales(MODELO_EMBEDDING)
almacen = Chroma(client=cliente, collection_name=COLECCION, embedding_function=embeddings)
denso = almacen.as_retriever(search_kwargs={"k": PROFUNDIDAD})
bm25 = BM25Retriever.from_documents(documentos, preprocess_func=tokenizar)
bm25.k = PROFUNDIDAD


def mezcla(pesos: list[float]) -> EnsembleRetriever:
    return EnsembleRetriever(retrievers=[denso, bm25], weights=pesos, c=C_RRF, id_key="chunk_id")


ensemble = mezcla([0.5, 0.5])

muestra = bm25.invoke("id_nubmer")[:3]
print("BM25 para el token id_nubmer")
for doc in muestra:
    print(f"  {doc.metadata['chunk_id'][:8]}  {doc.metadata['documento']}  p. {doc.metadata['pagina']}  {doc.metadata['seccion']}")
    print("   ", doc.page_content[:160].replace("\n", " "))


BM25 para el token id_nubmer
  aac71f95  Descriptor Guide  p. 18  Member objects and __slots__
    >>> auto.id_nubmer = 'VYE483814LQEX' Traceback (most recent call last): ... AttributeError: 'Vehicle' object has no attribute 'id_nubmer' 2. Helps create immuta
  88d7633a  Descriptor Guide  p. 18  Member objects and __slots__
    4.6 Member objects and __slots__ When a class defines __slots__, it replaces instance dictionaries with a fixed-length array of slot values. From a user point o
  8e24f43d  The Python Language Reference  p. iii  
    6.10.3 Identity comparisons . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 93 6.11 Boolean operations . . . . . . . . . . . . . . . . .


## 3. Quince preguntas y el fragmento correcto

Cinco llevan un token que casi no se repite: un ejemplo literal, un nombre, una función, una errata del propio manual y una bandera de `re`. Cinco son paráfrasis y no contienen las palabras del fragmento que las responde. Cinco mezclan un término concreto con una formulación más abierta.

El fragmento correcto es el que contiene la frase ancla. En dos preguntas la ancla cae en más de un fragmento; cuenta cualquiera de ellos. Recall@5 es el porcentaje de preguntas en las que ese fragmento aparece entre los cinco primeros.


In [3]:
PREGUNTAS = [
    {"id": "goodbye", "tipo": "específica", "pregunta": "Goodbye, world!", "ancla": "Goodbye, world!"},
    {"id": "lispers", "tipo": "específica", "pregunta": "paper intended for lispers", "ancla": "lispers"},
    {"id": "recursion", "tipo": "específica", "pregunta": "sys.setrecursionlimit", "ancla": "setrecursionlimit"},
    {"id": "nubmer", "tipo": "específica", "pregunta": "id_nubmer", "ancla": "id_nubmer"},
    {"id": "ignorecase", "tipo": "específica", "pregunta": "re.IGNORECASE", "ancla": "re.IGNORECASE"},
    {
        "id": "propiedad",
        "tipo": "semántica",
        "pregunta": "por qué un objeto no puede cambiar el comportamiento de un atributo que la clase calcula al leerlo",
        "ancla": "instances cannot override the behavior",
    },
    {
        "id": "pickle",
        "tipo": "semántica",
        "pregunta": "por qué es peligroso recuperar un objeto guardado en disco si quien lo escribió no es de confianza",
        "ancla": "untrusted source can execute arbitrary code",
    },
    {
        "id": "anidadas",
        "tipo": "semántica",
        "pregunta": "qué ocurre con los nombres que una función de dentro usa pero que fueron creados en la función de fuera",
        "ancla": "Free variables used in the nested function",
    },
    {
        "id": "limpieza",
        "tipo": "semántica",
        "pregunta": "cómo fuerzas un bloque de limpieza que se ejecuta tanto si el intento sale bien como si salta un error",
        "ancla": "define clean-up actions that must be executed",
    },
    {
        "id": "corchetes",
        "tipo": "semántica",
        "pregunta": "una manera breve de construir una secuencia nueva a partir de otra, escrita entre corchetes",
        "ancla": "List comprehensions provide a concise way",
    },
    {
        "id": "hash",
        "tipo": "mixta",
        "pregunta": "qué tiene que cumplir __hash__ para que dos objetos iguales sirvan de clave en un conjunto",
        "ancla": "objects which compare equal have the same hash",
    },
    {
        "id": "generador",
        "tipo": "mixta",
        "pregunta": "una generator expression frente a la lista equivalente, cuál ahorra memoria",
        "ancla": "parentheses instead of brackets",
    },
    {
        "id": "closed",
        "tipo": "mixta",
        "pregunta": "en el ejemplo de with open, qué valor tiene f.closed al salir del bloque",
        "ancla": "f.closed",
    },
    {
        "id": "dylan",
        "tipo": "mixta",
        "pregunta": "el method resolution order que Python 2.3 copió de Dylan",
        "ancla": "people working on Dylan",
    },
    {
        "id": "yieldfrom",
        "tipo": "mixta",
        "pregunta": "diferencia de sintaxis entre yield from y un yield de una sola expresión",
        "ancla": "yield from <expr>",
    },
]


def relevantes(ancla: str) -> set[str]:
    return {doc.metadata["chunk_id"] for doc in documentos if ancla in doc.page_content}


def acierta(identificadores: list[str], buenos: set[str]) -> bool:
    return any(identificador in buenos for identificador in identificadores)


verdad = {pregunta["id"]: relevantes(pregunta["ancla"]) for pregunta in PREGUNTAS}
for pregunta in PREGUNTAS:
    print(f"{pregunta['id']:<12} {pregunta['tipo']:<12} fragmentos correctos: {len(verdad[pregunta['id']])}")


goodbye      específica   fragmentos correctos: 1
lispers      específica   fragmentos correctos: 1
recursion    específica   fragmentos correctos: 1
nubmer       específica   fragmentos correctos: 2
ignorecase   específica   fragmentos correctos: 3
propiedad    semántica    fragmentos correctos: 1
pickle       semántica    fragmentos correctos: 1
anidadas     semántica    fragmentos correctos: 1
limpieza     semántica    fragmentos correctos: 1
corchetes    semántica    fragmentos correctos: 1
hash         mixta        fragmentos correctos: 1
generador    mixta        fragmentos correctos: 1
closed       mixta        fragmentos correctos: 1
dylan        mixta        fragmentos correctos: 1
yieldfrom    mixta        fragmentos correctos: 4


## 4. Top 5 y Recall@5

La marca `sí` significa que el fragmento ancla está entre esos cinco. Los identificadores van abreviados.


In [4]:
def top_ids(docs: list[Document], n: int = K) -> list[str]:
    return [doc.metadata["chunk_id"] for doc in docs[:n]]


def evaluar(recuperador) -> list[dict]:
    filas = []
    for pregunta in PREGUNTAS:
        docs = recuperador.invoke(pregunta["pregunta"])
        ids = top_ids(docs)
        filas.append({"pregunta": pregunta, "ids": ids, "acierto": acierta(ids, verdad[pregunta["id"]])})
    return filas


def recall(filas: list[dict]) -> float:
    return sum(fila["acierto"] for fila in filas) / len(filas)


filas_densas = evaluar(denso)
filas_bm25 = evaluar(bm25)
filas_hibridas = evaluar(ensemble)

print(f"{'id':<12} {'tipo':<12} {'denso':<6} {'bm25':<6} {'híbrido'}")
for a, b, c in zip(filas_densas, filas_bm25, filas_hibridas):
    pregunta = a["pregunta"]
    print(
        f"{pregunta['id']:<12} {pregunta['tipo']:<12} "
        f"{'sí' if a['acierto'] else 'no':<6} "
        f"{'sí' if b['acierto'] else 'no':<6} "
        f"{'sí' if c['acierto'] else 'no'}"
    )
print()
print(f"Recall@5  denso {recall(filas_densas):.0%}   BM25 {recall(filas_bm25):.0%}   híbrido 0.5/0.5 {recall(filas_hibridas):.0%}")
print()
print("Top 5 abreviados. El asterisco marca un id correcto.")
for a, b, c in zip(filas_densas, filas_bm25, filas_hibridas):
    pregunta = a["pregunta"]
    buenos = verdad[pregunta["id"]]

    def pintar(ids: list[str]) -> str:
        return " ".join(("*" if identificador in buenos else "") + identificador[:6] for identificador in ids)

    print(f"{pregunta['id']:<12} D {pintar(a['ids'])}")
    print(f"{'':<12} B {pintar(b['ids'])}")
    print(f"{'':<12} H {pintar(c['ids'])}")


id           tipo         denso  bm25   híbrido
goodbye      específica   no     sí     sí
lispers      específica   no     sí     sí
recursion    específica   sí     sí     sí
nubmer       específica   no     sí     sí
ignorecase   específica   sí     sí     sí
propiedad    semántica    sí     no     sí
pickle       semántica    sí     no     sí
anidadas     semántica    no     no     no
limpieza     semántica    sí     no     sí
corchetes    semántica    sí     no     no
hash         mixta        sí     no     sí
generador    mixta        sí     sí     sí
closed       mixta        sí     sí     sí
dylan        mixta        sí     sí     sí
yieldfrom    mixta        sí     sí     sí

Recall@5  denso 73%   BM25 60%   híbrido 0.5/0.5 87%

Top 5 abreviados. El asterisco marca un id correcto.
goodbye      D 426200 512bbf 47ddb1 cdc53d 9cefeb
             B *20eacc 210f04 7de97b 4afb52 a7a159
             H 426200 *20eacc 512bbf 210f04 47ddb1
lispers      D 5c8cab 4b5665 aae14a 5afbf8 8c88

## 5. Otros pesos

0.7/0.3 inclina la fusión hacia el denso. 0.3/0.7 la inclina hacia BM25. Los candidatos son los mismos; solo cambia el peso del rango.


In [5]:
def fusionar(listas: list[list[Document]], pesos: list[float]) -> list[Document]:
    puntajes: dict[str, float] = {}
    elegido: dict[str, Document] = {}
    for lista, peso in zip(listas, pesos):
        for rango, doc in enumerate(lista, start=1):
            clave = doc.metadata["chunk_id"]
            puntajes[clave] = puntajes.get(clave, 0.0) + peso / (rango + C_RRF)
            elegido.setdefault(clave, doc)
    return [elegido[clave] for clave in sorted(puntajes, key=puntajes.get, reverse=True)]


candidatos = {}
for pregunta in PREGUNTAS:
    candidatos[pregunta["id"]] = [
        denso.invoke(pregunta["pregunta"]),
        bm25.invoke(pregunta["pregunta"]),
    ]

print(f"{'pesos denso/BM25':<20} {'Recall@5':>8}  {'específicas':>12} {'semánticas':>11} {'mixtas':>7}")
for pesos in ([0.5, 0.5], [0.7, 0.3], [0.3, 0.7]):
    filas = []
    for pregunta in PREGUNTAS:
        ids = top_ids(fusionar(candidatos[pregunta["id"]], pesos))
        filas.append({"pregunta": pregunta, "acierto": acierta(ids, verdad[pregunta["id"]])})
    por_tipo = {}
    for tipo in ("específica", "semántica", "mixta"):
        del_tipo = [fila for fila in filas if fila["pregunta"]["tipo"] == tipo]
        por_tipo[tipo] = sum(fila["acierto"] for fila in del_tipo) / len(del_tipo)
    print(
        f"{pesos[0]:.1f}/{pesos[1]:.1f}{'':<14} {recall(filas):>8.0%}  "
        f"{por_tipo['específica']:>12.0%} {por_tipo['semántica']:>11.0%} {por_tipo['mixta']:>7.0%}"
    )


pesos denso/BM25     Recall@5   específicas  semánticas  mixtas
0.5/0.5                    87%          100%         60%    100%
0.7/0.3                    87%           80%         80%    100%
0.3/0.7                    67%          100%          0%    100%


## 6. RRF a mano

La función de arriba es la misma cuenta que `EnsembleRetriever`: `peso / (rango + 60)`, desempate por identificador. Con pesos 0.5/0.5 el top 5 tiene que coincidir con el del ensemble.


In [6]:
coinciden = 0
for pregunta, fila in zip(PREGUNTAS, filas_hibridas):
    manual = top_ids(fusionar(candidatos[pregunta["id"]], [0.5, 0.5]))
    if manual == fila["ids"]:
        coinciden += 1
    else:
        print(pregunta["id"], "ensemble", [i[:6] for i in fila["ids"]], "manual", [i[:6] for i in manual])
print(f"Top 5 idéntico en {coinciden}/{len(PREGUNTAS)} preguntas")


Top 5 idéntico en 15/15 preguntas


## 7. La cadena, con el ensemble en el sitio del denso

Misma cadena LCEL que en 3.1. El recuperador ahora es la mezcla 0.5/0.5. Dos preguntas bastan para ver que sigue respondiendo: una con el token literal y una paráfrasis.


In [7]:
def formatear(docs: list[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)


prompt = ChatPromptTemplate.from_template(
    "Responde a la pregunta usando solo el contexto. "
    "Si el contexto no contiene la respuesta, di que no lo sabes. No inventes.\n\n"
    "Contexto:\n{context}\n\n"
    "Pregunta: {question}"
)
llm = ChatOllama(model=MODELO_LLM, temperature=0, num_ctx=8192, num_predict=180, keep_alive=1800)
cadena = (
    {"context": ensemble | formatear, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
for texto in (
    "Qué se imprime en el ejemplo que contiene Goodbye, world! y KeyboardInterrupt?",
    "por qué es peligroso recuperar un objeto guardado en disco si quien lo escribió no es de confianza",
):
    print("=" * 70)
    print(texto)
    print(cadena.invoke(texto).strip())


Qué se imprime en el ejemplo que contiene Goodbye, world! y KeyboardInterrupt?
En el ejemplo que contiene "Goodbye, world!" y KeyboardInterrupt, se imprime "Goodbye, world!" antes de que se muestre el error de KeyboardInterrupt. Esto se debe a que el finally clause se ejecuta antes de que se re-raise el error.
por qué es peligroso recuperar un objeto guardado en disco si quien lo escribió no es de confianza
Es peligroso recuperar un objeto guardado en disco si quien lo escribió no es de confianza porque, según el texto, "deserializing pickle data coming from an untrusted source can execute arbitrary code, if the data was crafted by a skilled attacker". Esto significa que si recuperas un objeto guardado en disco que fue escrito por alguien que no es de confianza, podrías ejecutar código arbitrario en tu sistema, lo que podría ser peligroso.


## Bonus 1 · Hybrid nativo en Qdrant

Los mismos vectores densos salen de Chroma, sin pasar otra vez por Ollama. Qdrant calcula el BM25 disperso con `Qdrant/bm25` y fusiona los dos prefetch con RRF. El cliente es local, en `qdrant_db/`. La latencia que se compara es la de la búsqueda, con el vector de la pregunta ya calculado.


In [8]:
qdrant = QdrantClient(path=str(QDRANT_DIR))
existe = qdrant.collection_exists(COLECCION_QDRANT)
puntos = qdrant.count(COLECCION_QDRANT).count if existe else 0
if puntos != len(documentos):
    if existe:
        qdrant.delete_collection(COLECCION_QDRANT)
    qdrant.create_collection(
        collection_name=COLECCION_QDRANT,
        vectors_config={"dense": models.VectorParams(size=1024, distance=models.Distance.COSINE)},
        sparse_vectors_config={"bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)},
    )
    crudos = cruda.get(include=["embeddings"])
    vectores = [vector.tolist() for vector in crudos["embeddings"]]
    orden = {identificador: posicion for posicion, identificador in enumerate(crudos["ids"])}
    lote = 64
    for inicio in range(0, len(documentos), lote):
        puntos_lote = []
        for doc in documentos[inicio : inicio + lote]:
            puntos_lote.append(
                models.PointStruct(
                    id=doc.metadata["chunk_id"],
                    vector={
                        "dense": vectores[orden[doc.metadata["chunk_id"]]],
                        "bm25": models.Document(text=doc.page_content, model="Qdrant/bm25"),
                    },
                    payload=doc.metadata,
                )
            )
        qdrant.upsert(COLECCION_QDRANT, points=puntos_lote)
        print(f"  qdrant {min(inicio + lote, len(documentos))}/{len(documentos)}", flush=True)
else:
    print(f"Qdrant reutiliza {puntos} puntos")


def buscar_qdrant(pregunta: str) -> list[str]:
    vector = embeddings.embed_query(pregunta)
    resultado = qdrant.query_points(
        collection_name=COLECCION_QDRANT,
        prefetch=[
            models.Prefetch(query=vector, using="dense", limit=PROFUNDIDAD),
            models.Prefetch(
                query=models.Document(text=pregunta, model="Qdrant/bm25"),
                using="bm25",
                limit=PROFUNDIDAD,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=K,
    )
    return [str(punto.id) for punto in resultado.points]


for pregunta in PREGUNTAS:
    embeddings.embed_query(pregunta["pregunta"])

tiempos_langchain = []
tiempos_qdrant = []
aciertos_qdrant = 0
for pregunta in PREGUNTAS:
    inicio = time.perf_counter()
    ids_langchain = top_ids(ensemble.invoke(pregunta["pregunta"]))
    tiempos_langchain.append(time.perf_counter() - inicio)
    inicio = time.perf_counter()
    ids_qdrant = buscar_qdrant(pregunta["pregunta"])
    tiempos_qdrant.append(time.perf_counter() - inicio)
    if acierta(ids_qdrant, verdad[pregunta["id"]]):
        aciertos_qdrant += 1

print(f"Recall@5  LangChain híbrido {recall(filas_hibridas):.0%}   Qdrant RRF {aciertos_qdrant / len(PREGUNTAS):.0%}")
print(
    f"Latencia media de la búsqueda  LangChain {sum(tiempos_langchain) / len(tiempos_langchain) * 1000:.0f} ms   "
    f"Qdrant {sum(tiempos_qdrant) / len(tiempos_qdrant) * 1000:.0f} ms"
)


Qdrant reutiliza 2698 puntos
Recall@5  LangChain híbrido 87%   Qdrant RRF 87%
Latencia media de la búsqueda  LangChain 6 ms   Qdrant 48 ms


## Bonus 2 · Lematización española

Este corpus está en inglés. `es_core_news_sm` lematizaría `yield`, `__hash__` y `PEP` como si fueran palabras españolas y rompería justo los tokens que hacen útil a BM25. No se indexa un segundo BM25 con ese modelo.

En un corpus en español el mismo paso sí ayuda: `funciones` y `función` pasan a la misma forma y BM25 deja de tratarlas como tokens distintos. Aquí esa ganancia no existe, y el coste es perder identificadores.


## Bonus 3 · Dónde gana cada uno

Recall@5 con pesos 0.5/0.5: denso 73 %, BM25 60 %, híbrido 87 %. La mezcla se queda los aciertos de los dos y solo pierde uno.

**La mezcla mejora al denso.** `Goodbye, world!`, `lispers` e `id_nubmer`. El denso no los trae en el top 5: son un literal, un nombre que sale una vez y una errata del manual. BM25 los pone en el puesto 1 y la fusión los conserva. Es el caso para el que existe el híbrido.

**Van parejos.** `sys.setrecursionlimit`, `re.IGNORECASE` y `f.closed`. El token es raro y además está escrito en un párrafo que el embedding entiende. Las tres listas marcan el fragmento, y en `f.closed` las tres lo ponen primero.

**El denso funciona mejor.** La paráfrasis de las list comprehensions está en el puesto 5 del denso y la mezcla la expulsa: BM25 no tiene las palabras inglesas del fragmento y rellena el top con otros trozos. `__hash__` baja del puesto 1 al 4 porque BM25 arrastra otros fragmentos que contienen el token. El atributo calculado de `property` baja del 1 al 2 por el mismo ruido. En las funciones anidadas no gana nadie: ni el denso ni BM25 traen el párrafo de las free variables, y RRF no puede inventarlo.

**Pesos.** 0.5/0.5 y 0.7/0.3 empatan en el 87 % global. 0.7/0.3 sube las paráfrasis del 60 % al 80 % y baja las específicas del 100 % al 80 %. 0.3/0.7 deja las paráfrasis a cero: el peso léxico aplasta al denso. En este corpus el 0.5/0.5 es el que no suelta ningún token raro. Si las preguntas van a ser sobre todo conceptuales, 0.7/0.3 recupera la list comprehension sin caerse del todo en los nombres.

**Qdrant.** El mismo Recall@5, 87 %. La búsqueda, con el vector de la pregunta ya hecho, tarda 6 ms en la mezcla de LangChain y 47 ms en el RRF de Qdrant. Aquí el corpus cabe en memoria y Chroma más BM25 es más barato. Qdrant compensa cuando el índice ya no cabe en un proceso y se quiere la fusión dentro del motor.
